# Carga Historica de Nivel ANA

In [0]:
bronze_schema = spark.table("weather.bronze.nivel_ana").schema

historical_df = (
    spark.read
         .schema(bronze_schema)
         .option("multiLine", True)
         .json("/Volumes/weather/raw/ana_volume/json/serie_74100000.json")
)

historical_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("weather.bronze.nivel_ana")

In [0]:
%sql
select count(*) from weather.bronze.nivel_ana

# Historico anuales desde 1945

In [0]:
from pyspark.sql.functions import lit

json_path = "/Volumes/weather/raw/ana_volume/json/Histo/histo_serie_74100000.json"

df = (
    spark.read
    .option("multiline", "true")
    .json(json_path)
)

# Agregar columna fija
df = df.withColumn("Chuva_Adotada", lit("0"))
df = df.withColumn("Vazao_Adotada", lit("0"))

# asegurar orden de columnas según la tabla
columns = [
    "Chuva_Adotada",
    "Chuva_Adotada_Status",
    "Cota_Adotada",
    "Cota_Adotada_Status",
    "Data_Atualizacao",
    "Data_Hora_Medicao",
    "Vazao_Adotada",
    "Vazao_Adotada_Status",
    "codigoestacao"
]

df = df.select(columns)

(
    df.write
    .format("delta")
    .mode("append")
    .saveAsTable("weather.bronze.nivel_ana")
)


In [0]:
%sql
SELECT *
FROM weather.bronze.nivel_ana
WHERE codigoestacao = '74100000'
ORDER BY Data_Hora_Medicao
LIMIT 20;